# Lab 01: Send Your First Amazon Bedrock Request

**Day 1 - Session 1**

Goal: Build and verify a minimal request to Amazon Bedrock through its OpenAI-compatible API.

> Open this notebook in Google Colab:
> [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/bedrock-companion/blob/main/day1/labs/01-first-bedrock-request/start/01-first-bedrock-request.ipynb)

In [2]:
# Run this cell once to install the required package.
# If running locally with the course environment already activated, you can skip this.
!pip install "openai>=2.37.0" --quiet

In [3]:
import getpass
import re
from types import SimpleNamespace

from openai import OpenAI

## Section 1: Configure the Bedrock endpoint

Amazon Bedrock exposes an OpenAI-compatible endpoint for each AWS Region. Enter the instructor-approved Region and short-term key when prompted; `getpass` prevents the key from appearing in notebook output.

**Where to get the short-term API key:** In the AWS Console, go to **Amazon Bedrock → API keys → Short-term API keys** tab and click **Generate short-term API keys** (see `short-term-api-keys.png` in this lab folder). In the dialog that opens, click **Copy API Key** to copy the `bedrock-api-key-...` value (see `copy-short-term-api-keys.png`), then paste it below. This is not the long-term IAM access key from your team's credentials file — the key you need here expires after 12 hours or when your console session ends.

In [4]:
DEFAULT_MODEL_ID = "openai.gpt-oss-20b-1:0"
DEFAULT_PROMPT = "Please summarize the following customer support case for a retail return:\n\nCustomer: I received the wrong size for my shoes (Order #12345). I ordered a size 10 but received a size 8.\nAction required: Summarize the issue and suggest the next steps for the agent."

aws_region = input("Instructor-approved AWS Region: ").strip()
bedrock_api_key = getpass.getpass("Short-term Amazon Bedrock API key: ").strip()

if not aws_region or not re.fullmatch(r"[a-z]{2}(?:-gov)?-[a-z]+-\d", aws_region):
    raise ValueError("Enter a valid AWS Region, such as us-east-1")
if not bedrock_api_key:
    raise ValueError("Amazon Bedrock API key must not be empty")

base_url = f"https://bedrock-runtime.{aws_region}.amazonaws.com/openai/v1"
client = OpenAI(api_key=bedrock_api_key, base_url=base_url)
print(f"Configured Amazon Bedrock in {aws_region}; key was not displayed.")

Configured Amazon Bedrock in us-east-1; key was not displayed.


## Section 2: Implement the request function

Keep request logic separate from configuration by accepting an injected client. The function must reject blank input, make exactly one Chat Completions call, remove any leading reasoning block, and reject empty output.

In [5]:
def ask_bedrock(client, prompt, model_id=DEFAULT_MODEL_ID):
    """Send one user prompt and return a non-empty answer."""
    # TODO: Reject a blank prompt with ValueError("prompt must not be empty").
    # TODO: Call client.chat.completions.create exactly once with model_id and one user message.
    # TODO: Read response.choices[0].message.content and remove any <reasoning>...</reasoning> block.
    # TODO: Strip whitespace and raise RuntimeError containing "empty text response" if no text remains.
    # TODO: Return the non-empty answer.
    raise NotImplementedError("Complete the TODOs in ask_bedrock")

## Section 3: Run an offline acceptance check

This fake client records the request without using credentials or model tokens. Passing this check confirms the request shape and response cleanup before the live call.

In [6]:
class FakeCompletions:
    def __init__(self):
        self.calls = []

    def create(self, **kwargs):
        self.calls.append(kwargs)
        message = SimpleNamespace(content="<reasoning>private work</reasoning>  Test answer.  ")
        return SimpleNamespace(choices=[SimpleNamespace(message=message)])


fake_completions = FakeCompletions()
fake_client = SimpleNamespace(
    chat=SimpleNamespace(completions=fake_completions),
)
answer = ask_bedrock(fake_client, "What is Amazon Bedrock?", "test-model")

assert answer == "Test answer."
assert fake_completions.calls == [{
    "model": "test-model",
    "messages": [{"role": "user", "content": "What is Amazon Bedrock?"}],
}]
print("OFFLINE_CHECK_OK")

NotImplementedError: Complete the TODOs in ask_bedrock

## Section 4: Send one live request

The response wording is nondeterministic. Success means that the call returns non-empty text and the notebook prints the marker below.

In [ ]:
live_answer = ask_bedrock(client, DEFAULT_PROMPT)
print(live_answer)
print(f"\nMODEL_RESPONSE_OK model={DEFAULT_MODEL_ID}")

## Cleanup

Restart the Colab runtime or Jupyter kernel now to clear the short-term key from memory. Do not save executed output that could contain sensitive error details.

In [ ]:
print("Lab complete.")
print("Takeaway: An injected client keeps Bedrock request logic testable while endpoint and credential configuration remain separate.")